In [ ]:
# Imports
import glob
import os
import string
import sys
import tarfile

# Path setup
sys.path.append("/data/workspace/lexpredict-contraxsuite-core/")

# Packages
from lexnlp.nlp.en.segments.sentences import get_sentence_list
from lexnlp.nlp.en.tokens import get_stems

import numpy
import pandas


# Gensim
import gensim.models.word2vec
import gensim.models.doc2vec

# Sklearn
import sklearn.linear_model
import sklearn.ensemble

In [ ]:
def process_sentence(sentence):
    sentence_stems = [s for s in get_stems(sentence, stopword=True, lowercase=True) if s.isalpha()]
    return sentence_stems

def process_document(document):
    doc_words = []
    for sentence in get_sentence_list(document):
        doc_words.extend(process_sentence(sentence))
    return doc_words

In [ ]:
#d2v_model = gensim.models.word2vec.Word2Vec.load("../models/d2v_cbow_lease_size200_window10")
d2v_model = gensim.models.word2vec.Word2Vec.load("../models/d2v_all_size200_window20")

In [ ]:
# Load coding file
coding_df = pandas.read_csv("../all_coding.csv", low_memory=False)

In [ ]:
# Sample data
doc_vectors = []
doc_targets = []
min_stem_count = 10

# Iterate through class paths
file_name = "/data/workspace/lexpredict-contraxsuite-core/test_data/agreements-text.tar.gz"

with tarfile.open(file_name, "r:gz") as corpus_tar_file:
    member_list = corpus_tar_file.getmembers()[0:10000]
    num_members = len(member_list)
    
    # Iterate through all
    for i, tar_member in enumerate(member_list):
        # Output
        if i % 100 == 0:
            print((file_name, i, float(i)/num_members * 100., tar_member.name, len(doc_vectors)))

        # Get class from coding file
        member_base_name = os.path.basename(tar_member.name)
        try:
            class_name = coding_df.loc[coding_df["output_file_name"] == member_base_name, "class"].values[0]
        except IndexError:
            continue
        
        # Read file
        member_file = corpus_tar_file.extractfile(tar_member.name)
        if member_file is None:
            print((file_name, tar_member.name, "invalid file"))
            continue
        member_buffer = member_file.read().decode("utf-8")
        if len(member_buffer.strip()) == 0:
            continue

        # Parse into sentence data
        try:
            doc_words = process_document(member_buffer)
            doc_vectors.append(d2v_model.infer_vector(doc_words))
            doc_targets.append(class_name)            
        except Exception as e:
            print(e)

In [ ]:
# Test logistic model
log_model = sklearn.linear_model.LogisticRegressionCV()
log_model.fit(doc_vectors, doc_targets)

# Test in-sample
log_predicted = log_model.predict(doc_vectors)
print(sklearn.metrics.classification_report(doc_targets, log_predicted))

In [ ]:
# Test logistic model
rf_model = sklearn.ensemble.RandomForestClassifier(n_estimators=100)
#rf_model = sklearn.ensemble.ExtraTreesClassifier(n_estimators=100)
rf_model.fit(doc_vectors, doc_targets)

# Test in-sample
rf_predicted = rf_model.predict(doc_vectors)
print(sklearn.metrics.classification_report(doc_targets, rf_predicted))

In [ ]:
# Sample data
test_doc_vectors = []
test_doc_targets = []
min_stem_count = 10

# Iterate through class paths
file_name = "/data/workspace/lexpredict-contraxsuite-core/test_data/agreements-text.tar.gz"

with tarfile.open(file_name, "r:gz") as corpus_tar_file:
    member_list = corpus_tar_file.getmembers()[15000:16000]
    num_members = len(member_list)
    
    # Iterate through all
    for i, tar_member in enumerate(member_list):
        # Output
        if i % 100 == 0:
            print((file_name, i, float(i)/num_members * 100., tar_member.name, len(test_doc_vectors)))

        # Get class from coding file
        member_base_name = os.path.basename(tar_member.name)
        try:
            class_name = coding_df.loc[coding_df["output_file_name"] == member_base_name, "class"].values[0]
        except IndexError:
            continue
        
        # Read file
        member_file = corpus_tar_file.extractfile(tar_member.name)
        if member_file is None:
            print((file_name, tar_member.name, "invalid file"))
            continue
        member_buffer = member_file.read().decode("utf-8")
        if len(member_buffer.strip()) == 0:
            continue

        # Parse into sentence data
        try:
            test_doc_words = process_document(member_buffer)
            test_doc_vectors.append(d2v_model.infer_vector(test_doc_words))
            test_doc_targets.append(class_name)            
        except Exception as e:
            print(e)

In [ ]:
# Predict
log_test_doc_predicted = log_model.predict(test_doc_vectors).tolist()
rf_test_doc_predicted = rf_model.predict(test_doc_vectors).tolist()

# Output report
print("Logistic:")
print(sklearn.metrics.classification_report(test_doc_targets, log_test_doc_predicted))

print("Random Forest:")
print(sklearn.metrics.classification_report(test_doc_targets, rf_test_doc_predicted))